# Anime data preparation — Member 1

## Goal

Build and read-back verify the four canonical PySpark tables used by Member 2. The notebook is an orchestration and quality-check artifact; transformation logic lives in tested modules under `src/`.

## Setup

Run this notebook from the repository root. It uses explicit feedback only for ALS, keeps `rating=-1` separately, uses seed 42, and prohibits `community_rating` leakage.

In [1]:
from pathlib import Path
import json
import os
import sys

START_DIR = Path.cwd()
ROOT = START_DIR if (START_DIR / "src").exists() else START_DIR.parent
if not (ROOT / "src").exists() or not (ROOT / "database").exists():
    raise RuntimeError("Run this notebook from the repository root")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

from src.config import RAW_RATINGS_PATH, RAW_ANIME_PATH, PROCESSED_DIR, SEED
from src.pipeline import create_spark, run_pipeline

for required in (RAW_RATINGS_PATH, RAW_ANIME_PATH):
    if not required.exists():
        raise FileNotFoundError(f"Required raw input is missing: {required}")
print({"seed": SEED, "ratings": str(RAW_RATINGS_PATH), "anime": str(RAW_ANIME_PATH)})

{'seed': 42, 'ratings': 'C:\\Users\\Quoc\\Desktop\\DTU\\ML Large Data\\Anime Recommend\\database\\rating.csv', 'anime': 'C:\\Users\\Quoc\\Desktop\\DTU\\ML Large Data\\Anime Recommend\\database\\anime.csv'}


## Steps

### 1. Run the declared-schema pipeline

The pipeline profiles raw inputs, normalizes catalogue keys/text, classifies ratings, aggregates duplicate pairs, removes orphans, validates reconciliation, writes each Parquet table atomically, and reads every output back.

In [2]:
spark = create_spark()
manifest = run_pipeline(spark)
print(json.dumps({k: v["rows"] for k, v in manifest["outputs"].items()}, indent=2))

{
  "clean_ratings": 6337232,
  "watched_unrated": 1476488,
  "clean_anime": 12294,
  "genre_features": 12294
}


### 2. Inspect the long-format audit

In [3]:
import pandas as pd
audit = pd.read_csv(PROCESSED_DIR / "data_quality_summary.csv")
audit

,run_id,stage,entity,metric,value,unit,rule_or_note
0,20260712T040205Z,clean,anime,community_rating_missing,230.0,rows,missing/out-of-range aggregate rating
1,20260712T040205Z,clean,anime,distinct_items,12294.0,items,unique normalized keys
2,20260712T040205Z,clean,anime,episodes_missing,340.0,rows,unparsable/missing/non-positive episodes
3,20260712T040205Z,clean,anime,members_missing,0.0,rows,invalid/missing members
4,20260712T040205Z,clean,anime,row_count,12294.0,rows,normalized catalog
...,...,...,...,...,...,...,...
63,20260712T040205Z,raw,ratings,rating_value_6_rows,637775.0,rows,raw rating=6
64,20260712T040205Z,raw,ratings,rating_value_7_rows,1375287.0,rows,raw rating=7
65,20260712T040205Z,raw,ratings,rating_value_8_rows,1646019.0,rows,raw rating=8
66,20260712T040205Z,raw,ratings,rating_value_9_rows,1254096.0,rows,raw rating=9


## Checks

The following cell fails if the output validation recorded anything other than a clean model-input contract.

In [4]:
assert manifest["validation"]["passed"] is True
assert manifest["validation"]["duplicate_pairs"] == 0
assert manifest["validation"]["invalid_rows"] == 0
assert manifest["validation"]["orphan_rows"] == 0
print("All canonical output and read-back quality gates passed.")

All canonical output and read-back quality gates passed.


## Next steps

Member 2 splits only `clean_ratings` with seed 42, then runs `validate_split`. Split summaries, cold-start exclusions, and train-only popularity are intentionally deferred until that real split exists.

In [5]:
spark.stop()